# Notebook 14 — Validación cruzada trilingüe Python + R + Julia

Demuestra de manera operativa el carácter multilingüe del proyecto ejecutando el mismo análisis de correlación caudal-NDVI en los tres lenguajes del curso ---Python, R y Julia--- y comparando los resultados pixel a pixel. La hipótesis a verificar es que la metodología del proyecto produce los mismos coeficientes de correlación de Pearson independientemente del lenguaje en que se implemente, en este sentido la convergencia numérica entre los tres flujos constituye una validación cruzada robusta de las conclusiones del informe sobre el acoplamiento hidroclimático del manglar de la CGSM.

**Insumos:**
- `data/raw/ideam/descargaDhime_elbanco_medio.csv` (caudal medio mensual El Banco, IDEAM)
- `outputs/tables/serie_temporal_ndvi_definitiva.csv` (NDVI mensual de las 8 estaciones)
- `src/R/08_correlacion_trilingual.R` (componente R, invocado vía `Rscript`)
- `src/julia/05_correlacion_trilingual.jl` (componente Julia, invocado vía `julia`)

**Productos:**
- `outputs/tables/validacion_trilingual_resultados.csv`
- `outputs/figures/validacion_trilingual_correlacion.png`

In [ ]:
import subprocess
from io import StringIO
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path('..').resolve()
OUT_TAB = ROOT / 'outputs' / 'tables' / 'validacion_trilingual_resultados.csv'
OUT_FIG = ROOT / 'outputs' / 'figures' / 'validacion_trilingual_correlacion.png'
OUT_TAB.parent.mkdir(parents=True, exist_ok=True)
OUT_FIG.parent.mkdir(parents=True, exist_ok=True)
print('Trilingual setup OK')

## 1. Componente Python — análisis de referencia

Replica la lógica del notebook 12b restringida a la correlación El Banco vs NDVI manglar.

In [ ]:
caudal = pd.read_csv(ROOT / 'data' / 'raw' / 'ideam' / 'descargaDhime_elbanco_medio.csv')
caudal['date'] = pd.to_datetime(caudal['Fecha']) + pd.offsets.Day(14)
caudal['caudal'] = pd.to_numeric(caudal['Valor'], errors='coerce')
caudal = caudal.dropna(subset=['caudal']).copy()
caudal['mes'] = caudal['date'].dt.month
clim = caudal.groupby('mes')['caudal'].agg(['mean', 'std']).reset_index()
caudal = caudal.merge(clim, on='mes')
caudal['caudal_z'] = (caudal['caudal'] - caudal['mean']) / caudal['std']
caudal_q = caudal[['date', 'caudal_z']].sort_values('date').reset_index(drop=True)

manglar = {'Cano_Palos', 'Cano_Clarin', 'CP_Aguas_Negras', 'CP_Luna'}
ndvi = pd.read_csv(ROOT / 'outputs' / 'tables' / 'serie_temporal_ndvi_definitiva.csv',
                   parse_dates=['date'])
ndvi = ndvi[ndvi['subzona'].isin(manglar)].copy()
ndvi['z'] = ndvi.groupby('subzona')['ndvi'].transform(
    lambda x: (x - x.mean()) / x.std())
ndvi['date_m'] = ndvi['date'].dt.to_period('M').dt.to_timestamp() + pd.offsets.Day(14)
z_mensual = ndvi.groupby('date_m')['z'].mean().reset_index().rename(columns={'date_m': 'date'})

merged = z_mensual.merge(caudal_q, on='date', how='inner').sort_values('date').reset_index(drop=True)

filas_py = []
for lag in range(0, 4):
    caudal_lag = merged['caudal_z'].shift(lag)
    validos = (~caudal_lag.isna()) & (~merged['z'].isna())
    rho = merged.loc[validos, 'z'].corr(caudal_lag[validos])
    filas_py.append({'lenguaje': 'Python', 'rezago_meses': lag,
                     'rho_caudal': round(rho, 4), 'n': int(validos.sum())})

df_py = pd.DataFrame(filas_py)
print('=== Python ===')
print(df_py.to_string(index=False))

## 2. Componente R — invocación vía `Rscript`

In [ ]:
r_script = ROOT / 'src' / 'R' / '08_correlacion_trilingual.R'
result = subprocess.run(['Rscript', str(r_script)],
                        capture_output=True, text=True)
if result.returncode != 0:
    print('ERROR R:', result.stderr)
    df_r = pd.DataFrame()
else:
    df_r = pd.read_csv(StringIO(result.stdout))
    print('=== R ===')
    print(df_r.to_string(index=False))

## 3. Componente Julia — invocación vía `julia`

In [ ]:
jl_script = ROOT / 'src' / 'julia' / '05_correlacion_trilingual.jl'
result = subprocess.run(['julia', str(jl_script)],
                        capture_output=True, text=True)
if result.returncode != 0:
    print('ERROR Julia:', result.stderr)
    df_jl = pd.DataFrame()
else:
    # Filtrar solo las líneas CSV (descartar otros prints de Julia)
    lineas = [l for l in result.stdout.split('\n')
              if l.startswith('lenguaje,') or l.startswith('Julia,')]
    df_jl = pd.read_csv(StringIO('\n'.join(lineas)))
    print('=== Julia ===')
    print(df_jl.to_string(index=False))

## 4. Tabla consolidada de las tres implementaciones

In [ ]:
df_all = pd.concat([df_py, df_r, df_jl], ignore_index=True)
df_wide = df_all.pivot(index='rezago_meses', columns='lenguaje', values='rho_caudal')
df_wide['Δ max'] = df_wide.max(axis=1) - df_wide.min(axis=1)
print('=== Convergencia trilingüe ===')
print(df_wide.round(4))

df_all.to_csv(OUT_TAB, index=False)
print(f'\nGuardado: {OUT_TAB}')
print(f'\nDiferencia máxima entre los tres lenguajes: {df_wide["Δ max"].max():.6f}')

## 5. Figura comparativa: ρ por lenguaje y rezago

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(4); w = 0.27
colors = {'Python': '#3776AB', 'R': '#276DC3', 'Julia': '#9558B2'}
for i, lang in enumerate(['Python', 'R', 'Julia']):
    sub = df_all[df_all.lenguaje == lang].sort_values('rezago_meses')
    if len(sub) > 0:
        ax.bar(x + (i - 1) * w, sub['rho_caudal'], width=w,
               label=lang, color=colors[lang], alpha=0.85)
ax.axhline(0, color='black', lw=0.5)
ax.set_xticks(x); ax.set_xticklabels([f'{i} m' for i in range(4)])
ax.set_xlabel('Rezago temporal')
ax.set_ylabel(r'$\rho$ Pearson (caudal Magdalena vs NDVI manglar)')
ax.set_title('Validación cruzada trilingüe — caudal IDEAM El Banco vs NDVI manglar CGSM, 2013–2025',
             fontsize=11)
ax.legend(title='Lenguaje', fontsize=9); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(OUT_FIG, dpi=200, bbox_inches='tight')
plt.show()
print(f'Guardado: {OUT_FIG}')

## 6. Interpretación

Si las barras de Python, R y Julia coinciden visualmente para cada rezago ---y la diferencia máxima entre los tres lenguajes ($\Delta_{max}$) reportada en la tabla de convergencia es inferior a $10^{-3}$---, queda demostrado de manera operativa que el análisis de correlación caudal-NDVI del proyecto produce resultados idénticos en los tres lenguajes del curso, en este sentido las conclusiones del informe sobre el acoplamiento hidroclimático del manglar de la CGSM no dependen de la implementación ni del entorno computacional, así el pipeline multilingüe constituye una arquitectura interoperable robusta para el monitoreo costero. Pequeñas diferencias del orden de $10^{-4}$ son atribuibles únicamente a la precisión numérica interna de cada lenguaje y al redondeo en los cálculos intermedios de promedios y desviaciones estándar mensuales.